In [1]:
'''
task: FOL classification based on graph data from AST
model: GNN
dataset: P-FOLIO
evaluation: supervised fine-tuning
'''
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# load FOLIO dataset
import pandas as pd

fol_train = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/folio/data/pfolio_ast_train.csv")
fol_test = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/folio/data/pfolio_ast_test.csv")
fol_valid = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/folio/data/pfolio_ast_val.csv")

In [3]:
# logic to generate graph from AST
import json
import matplotlib.pyplot as plt
import networkx as nx

def extract_text(node):
    """Recursively processes specific sub-tokens to rebuild logic strings."""
    pieces = []
    if "text" in node:
        pieces.append(node["text"])
    if "children" in node:
        for child in node["children"]:
            pieces.extend(extract_text(child))
    return "".join(pieces)

def clean_syntax(text):
    """Formats string operations into formal math notation markers."""
    return (text.strip()
            .replace("forall", "∀ ")
            .replace("implies", " → ")
            .replace("and", " ∧ ")
            .replace("not", " ¬ ")
            .replace("()", ""))

def ast_to_syllogism_graph(ast_dict):
  """Transforms an AST into a directed NetworkX graph.

  - Group nodes with the same name into a single unique node.
  - Connect quantifiers to symbols.
  - Connect keywords to all terms in the statement.
  - Connect predicate terms to their argument terms within atomic propositions
    (e.g., japanese -> koeitecmo).
  """
  G = nx.DiGraph()

  # 1. Add Root Node
  root_node = "Syllogism"
  G.add_node(root_node, label="Syllogism", node_type="root")

  # Helper to safely locate statements in the AST structure
  def find_statements(node):
    stats = []
    if not isinstance(node, dict):
      return stats

    if node.get("type") == "stat":
      stats.append(node)

    for child in node.get("children", []):
      stats.extend(find_statements(child))

    return stats

  statements = find_statements(ast_dict)
  if not statements:
    return G

  premises = statements[:-1]
  conclusion = statements[-1]

  # Helper to split a term node's contents by commas (e.g., x,y or tecmo,japan)
  def extract_sub_terms_from_node(node):
    if not isinstance(node, dict):
      return []

    sub_terms = []
    current_letters = []
    for child in node.get("children", []):
      child_type = child.get("type")
      if child_type == "COMMA":
        if current_letters:
          sub_terms.append("".join(current_letters))
          current_letters = []
      elif child_type == "LETTER" and "text" in child:
        current_letters.append(child["text"])
      else:
        nested = extract_sub_terms_from_node(child)
        if nested:
          sub_terms.extend(nested)
    if current_letters:
      sub_terms.append("".join(current_letters))

    if not sub_terms and "text" in node:
      sub_terms.append(node["text"])

    return sub_terms

  # Helper to find text inside simple nodes like quantifier, keyword, symbol
  def get_node_text(node):
    if not isinstance(node, dict):
      return ""
    if "text" in node:
      return node["text"]
    for child in node.get("children", []):
      res = get_node_text(child)
      if res:
        return res
    return ""

  def collect_keywords(node, keywords):
    if not isinstance(node, dict):
      return
    if node.get("type") == "keyword":
      kw_text = get_node_text(node)
      if kw_text and kw_text not in keywords:
        keywords.append(kw_text)
    for child in node.get("children", []):
      collect_keywords(child, keywords)

  # Helper to find all atomic propositions in a subtree
  def find_atomic_propositions(node):
    aps = []
    if not isinstance(node, dict):
      return aps
    if node.get("type") == "atomicproposition":
      aps.append(node)
    for child in node.get("children", []):
      aps.extend(find_atomic_propositions(child))
    return aps

  # Helper to ensure unique nodes by name/label
  def add_unique_node(node_id, label, node_type):
    if not G.has_node(node_id):
      G.add_node(node_id, label=label, node_type=node_type)
    return node_id

  # Process a single statement (Premise or Conclusion)
  def process_statement(stat_node, parent_graph_node):
    # 1. Quantifier & Symbol
    quantifier_id = None
    symbol_id = None

    def find_quantifier_symbol(node):
      nonlocal quantifier_id, symbol_id
      if not isinstance(node, dict):
        return
      ntype = node.get("type")
      if ntype == "quantifier":
        q_text = get_node_text(node)
        if q_text:
          quantifier_id = add_unique_node(f"q_{q_text}", q_text, "quantifier")
          G.add_edge(parent_graph_node, quantifier_id)
      elif ntype == "symbol" and quantifier_id:
        s_text = get_node_text(node)
        if s_text:
          symbol_id = add_unique_node(f"s_{s_text}", s_text, "symbol")
          G.add_edge(quantifier_id, symbol_id)

      for child in node.get("children", []):
        find_quantifier_symbol(child)

    find_quantifier_symbol(stat_node)

    # 2. Process Atomic Propositions (Predicate -> Argument Term Edges)
    aps = find_atomic_propositions(stat_node)
    statement_terms = set()
    term_node_ids = {}

    for ap in aps:
      # Find direct term children of this atomic proposition
      term_children = [
          c for c in ap.get("children", []) if c.get("type") == "term"
      ]
      ap_terms = []
      for tc in term_children:
        sub = extract_sub_terms_from_node(tc)
        ap_terms.extend(sub)

      if ap_terms:
        # First term is the predicate/relation (e.g., japanese, disbandsin, holds)
        predicate_text = ap_terms[0]
        pred_id = add_unique_node(
            f"t_{predicate_text}", predicate_text, "term"
        )
        G.add_edge(parent_graph_node, pred_id)
        statement_terms.add(predicate_text)
        term_node_ids[predicate_text] = pred_id

        # Subsequent terms are arguments (e.g., koeitecmo, tecmo, japan, x, y)
        for arg_text in ap_terms[1:]:
          arg_id = add_unique_node(f"t_{arg_text}", arg_text, "term")
          G.add_edge(parent_graph_node, arg_id)
          statement_terms.add(arg_text)
          term_node_ids[arg_text] = arg_id

          # Create directed edge from predicate to argument term
          G.add_edge(pred_id, arg_id)

    # Fallback for any standalone terms not caught in atomic propositions
    def find_all_terms(node):
      if not isinstance(node, dict):
        return
      if node.get("type") == "term":
        for t in extract_sub_terms_from_node(node):
          if t not in statement_terms:
            t_id = add_unique_node(f"t_{t}", t, "term")
            G.add_edge(parent_graph_node, t_id)
            statement_terms.add(t)
            term_node_ids[t] = t_id
      for child in node.get("children", []):
        find_all_terms(child)

    find_all_terms(stat_node)

    # 3. Keywords -> Connect keyword to all terms in the statement
    keywords = []
    collect_keywords(stat_node, keywords)

    for kw in keywords:
      kw_id = add_unique_node(f"kw_{kw}", kw, "keyword")
      G.add_edge(parent_graph_node, kw_id)

      for t_text, t_id in term_node_ids.items():
        G.add_edge(kw_id, t_id)

  # 2. Process Premises
  for idx, premise in enumerate(premises):
    premise_name = f"Premise_{idx + 1}"
    add_unique_node(premise_name, premise_name, "premise")
    G.add_edge(root_node, premise_name)
    process_statement(premise, premise_name)

  # 3. Process Conclusion
  conclusion_name = "Conclusion"
  add_unique_node(conclusion_name, conclusion_name, "conclusion")
  G.add_edge(root_node, conclusion_name)
  process_statement(conclusion, conclusion_name)

  return G

In [4]:
# generate graphs from PFOLIO train dataset
train_graph_list = [] # graph transformation of each syllogism
train_label_list = [] # label for each graph (syllogism)
for idx, row in fol_train.iterrows():
    ast_data = []
    premises_ast = row["Premises - FOL"]
    conclusion_ast = row["Conclusions - FOL"]
    # replace single quotes with double quotes to make it valid JSON
    premises_ast = premises_ast.replace("'", '"')
    conclusion_ast = conclusion_ast.replace("'", '"')
    # add conlusion as a child to premises dict
    premises_ast = json.loads(premises_ast)
    conclusion_ast = json.loads(conclusion_ast)
    premises_ast["children"].append(conclusion_ast["children"][0]["children"][0])
    premises_ast = premises_ast["children"][0]  # get the first child of premises as the root of the AST
    ast_data.append(premises_ast)
    # generate graph from AST
    ast_graph = ast_to_syllogism_graph(ast_data[0])
    train_graph_list.append(ast_graph)
    train_label_list.append(row["Truth Values"])

print("*** Train data graph dataset generation complete ***")

# generate graphs from PFOLIO test dataset
test_graph_list = [] # graph transformation of each syllogism
test_label_list = [] # label for each graph (syllogism)
for idx, row in fol_test.iterrows():
    ast_data = []
    premises_ast = row["Premises - FOL"]
    conclusion_ast = row["Conclusions - FOL"]
    # replace single quotes with double quotes to make it valid JSON
    premises_ast = premises_ast.replace("'", '"')
    conclusion_ast = conclusion_ast.replace("'", '"')
    # add conlusion as a child to premises dict
    premises_ast = json.loads(premises_ast)
    conclusion_ast = json.loads(conclusion_ast)
    premises_ast["children"].append(conclusion_ast["children"][0]["children"][0])
    premises_ast = premises_ast["children"][0]  # get the first child of premises as the root of the AST
    ast_data.append(premises_ast)
    # generate graph from AST
    ast_graph = ast_to_syllogism_graph(ast_data[0])
    test_graph_list.append(ast_graph)
    test_label_list.append(row["Truth Values"])

print("*** Test data graph dataset generation complete ***")

# generate graphs from PFOLIO val dataset
val_graph_list = [] # graph transformation of each syllogism
val_label_list = [] # label for each graph (syllogism)
for idx, row in fol_valid.iterrows():
    ast_data = []
    premises_ast = row["Premises - FOL"]
    conclusion_ast = row["Conclusions - FOL"]
    # replace single quotes with double quotes to make it valid JSON
    premises_ast = premises_ast.replace("'", '"')
    conclusion_ast = conclusion_ast.replace("'", '"')
    # add conlusion as a child to premises dict
    premises_ast = json.loads(premises_ast)
    conclusion_ast = json.loads(conclusion_ast)
    premises_ast["children"].append(conclusion_ast["children"][0]["children"][0])
    premises_ast = premises_ast["children"][0]  # get the first child of premises as the root of the AST
    ast_data.append(premises_ast)
    # generate graph from AST
    ast_graph = ast_to_syllogism_graph(ast_data[0])
    val_graph_list.append(ast_graph)
    val_label_list.append(row["Truth Values"])

print("*** Val data graph dataset generation complete ***")

*** Train data graph dataset generation complete ***
*** Test data graph dataset generation complete ***
*** Val data graph dataset generation complete ***


In [5]:
!pip install scikit-learn torch-geometric
!pip install tensorboard

In [6]:
import torch
import networkx as nx
from torch_geometric.data import Data

def prepare_dataset_from_graphs(graphs_list, labels_list):
    """
    Converts a list of independent NetworkX DiGraphs into a list of PyTorch Geometric Data objects.

    Parameters:
    -----------
    graphs_list : list of nx.DiGraph
        A list where each element is a separate syllogism DiGraph (G).
    labels_list : list of int
        A list of integer labels corresponding to each graph in graphs_list.
        Values must be 0 (False), 1 (True), or 2 (Unknown).

    Returns:
    --------
    pyg_data_list : list of torch_geometric.data.Data
        A list of data objects ready for the PyTorch Geometric DataLoader.
    """
    # Map textual node types to unique integer categorical IDs
    type_map = {
        "root": 0,
        "premise": 1,
        "conclusion": 2,
        "quantifier": 3,
        "keyword": 4,
        "symbol": 5,
        "term": 6
    }

    pyg_data_list = []

    # Iterate through every graph-label pair in your dataset
    for idx, G in enumerate(graphs_list):
        node_list = list(G.nodes())
        node_to_idx = {node: i for i, node in enumerate(node_list)}

        # 1. Extract and map node category features
        node_features = []
        for node in node_list:
            if node.lower() == "syllogism":
                n_type = "root"
            elif node.lower() == "Conclusion":
                n_type = "conclusion"
            elif node.lower().startswith("P") and "_" not in node:
                n_type = "premise"
            elif "q" in node.lower():
                n_type = "quantifier"
            elif "keyword" in node.lower():
                n_type = "keyword"
            elif "symbol" in node.lower():
                n_type = "symbol"
            else:
                n_type = "term"

            node_features.append(type_map[n_type])

        # Convert node features to a long tensor shape [num_nodes, 1]
        x = torch.tensor(node_features, dtype=torch.long).view(-1, 1)

        # 2. Extract edge combinations safely if edges exist
        edges = list(G.edges())
        if len(edges) > 0:
            edge_index = torch.tensor(
                [[node_to_idx[u], node_to_idx[v]] for u, v in edges],
                dtype=torch.long
            ).t().contiguous()
        else:
            # Handle empty edge fallback edge index shape [2, 0]
            edge_index = torch.empty((2, 0), dtype=torch.long)

        # 3. Associate the structural classification target label
        y = torch.tensor([labels_list[idx]], dtype=torch.long)

        # 4. Enforce structural grouping into PyG Data format
        data_object = Data(x=x, edge_index=edge_index, y=y)
        pyg_data_list.append(data_object)

    return pyg_data_list


In [7]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, global_mean_pool
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from sklearn.model_selection import train_test_split

# ==========================================
# 1. EARLY STOPPING CLASS DEFINITION
# ==========================================
class EarlyStopping:
    """
    Stops training early if validation loss doesn't improve after a given patience.
    Saves a checkpoint of the best model state parameters.
    """
    def __init__(self, patience=5, min_delta=0.001, checkpoint_path="best_model.pt"):
        self.patience = patience
        self.min_delta = min_delta
        self.checkpoint_path = checkpoint_path
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.save_checkpoint(model)
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            print(f"EarlyStopping Counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.save_checkpoint(model)
            self.counter = 0

    def save_checkpoint(self, model):
        """Saves model weights when validation loss decreases."""
        torch.save(model.state_dict(), self.checkpoint_path)

# ==========================================
# 2. DATASET GENERATION GENERATOR
# ==========================================
class SyllogismDataset(Dataset):
    def __init__(self, num_samples=300):
        super().__init__()
        self.data_list = []
        mock_labels = np.random.choice([0, 1, 2], size=num_samples, p=[0.4, 0.3, 0.3])

        for idx in range(num_samples):
            num_nodes = 12
            x = torch.randint(0, 6, (num_nodes, 1), dtype=torch.long)
            sources = list(range(0, num_nodes - 1))
            targets = list(range(1, num_nodes))
            edge_index = torch.tensor([sources, targets], dtype=torch.long)
            y = torch.tensor([mock_labels[idx]], dtype=torch.long)
            self.data_list.append(Data(x=x, edge_index=edge_index, y=y))

    def len(self):
        return len(self.data_list)

    def get(self, idx):
        return self.data_list[idx]

# ==========================================
# 3. MODEL DEFINITION
# ==========================================
class SyllogismGNNClassifier(nn.Module):
    def __init__(self, num_node_types=6, embedding_dim=64, hidden_dim=64, num_classes=3):
        super().__init__()
        self.embedding = nn.Embedding(num_node_types, embedding_dim)
        self.conv1 = GATConv(embedding_dim, hidden_dim, heads=2, concat=True)
        self.conv2 = GATConv(hidden_dim * 2, hidden_dim, heads=1, concat=False)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = self.embedding(x.squeeze(-1))
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        graph_repr = global_mean_pool(x, batch)
        return self.fc(graph_repr)

# ==========================================
# 4. PIPELINE ORCHESTRATION WITH LOGGING
# ==========================================
def run_pipeline(train_graph_list, train_label_list, test_graph_list, test_label_list, val_graph_list, val_label_list):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Executing pipeline on device: {device}\n")

    # Initialize TensorBoard Writer
    writer = SummaryWriter(log_dir="runs/syllogism_classification")

    # Initialize Early Stopping
    early_stopping = EarlyStopping(patience=3, min_delta=0.001, checkpoint_path="best_syllogism_model.pt")

    # Initialize label mapping dictionary: ['0: F', '1: T', '2: U']
    label_map = {'F': 0, 'T': 1, 'U': 2}

    train_labels = [label_map[lbl] for lbl in train_label_list]
    test_labels = [label_map[lbl] for lbl in test_label_list]
    val_labels = [label_map[lbl] for lbl in val_label_list]

    # Assuming you have loaded your dataset variables:
    # my_networkx_graphs = [G1, G2, G3, ...]
    # my_target_labels   = [1,  0,  2,  ...]  # 1=True, 0=False, 2=Unknown

    # Step 1: Run your real networkx graphs through the updated function
    train_data = prepare_dataset_from_graphs(train_graph_list, train_labels)
    test_data = prepare_dataset_from_graphs(test_graph_list, test_labels)
    val_data = prepare_dataset_from_graphs(val_graph_list, val_labels)

    # Step 2: Pass directly into the PyG data loaders
    train_loader = DataLoader(train_data, batch_size=4)
    val_loader = DataLoader(test_data, batch_size=4)
    test_loader = DataLoader(val_data, batch_size=4)

    # Class weights tensor
    class_counts = np.bincount(train_labels, minlength=3)
    class_weights = 1.0 / (class_counts + 1e-6)
    class_weights = class_weights / np.sum(class_weights) * 3.0
    weight_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

    model = SyllogismGNNClassifier().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss(weight=weight_tensor)

    # Set high max epochs, rely on early stopping to close out training
    num_epochs = 10
    print("--- Beginning Training Loop with Logging & Early Stopping ---")

    for epoch in range(1, num_epochs + 1):
        model.train()
        total_train_loss, correct_train, total_train_graphs = 0, 0, 0

        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            out = model(batch)
            loss = criterion(out, batch.y)
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item() * batch.num_graphs
            preds = out.argmax(dim=-1)
            correct_train += int((preds == batch.y).sum())
            total_train_graphs += batch.num_graphs

        avg_train_loss = total_train_loss / total_train_graphs
        train_acc = correct_train / total_train_graphs

        # Validation Step
        model.eval()
        total_val_loss, correct_val, total_val_graphs = 0, 0, 0
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                out = model(batch)
                loss = criterion(out, batch.y)
                total_val_loss += loss.item() * batch.num_graphs
                preds = out.argmax(dim=-1)
                correct_val += int((preds == batch.y).sum())
                total_val_graphs += batch.num_graphs

        avg_val_loss = total_val_loss / total_val_graphs
        val_acc = correct_val / total_val_graphs

        print(f"Epoch {epoch:02d} | Train Loss: {avg_train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {avg_val_loss:.4f} Acc: {val_acc:.4f}")

        # Log to TensorBoard
        writer.add_scalar("Loss/Train", avg_train_loss, epoch)
        writer.add_scalar("Loss/Validation", avg_val_loss, epoch)
        writer.add_scalar("Accuracy/Train", train_acc, epoch)
        writer.add_scalar("Accuracy/Validation", val_acc, epoch)

        # Evaluate Early Stopping
        early_stopping(avg_val_loss, model)
        if early_stopping.early_stop:
            print(f"== Early stopping triggered at epoch {epoch}. Restoring best model weights. ==")
            break

    # Close TensorBoard logging process
    writer.close()

    # Load Best Model Checkpoint for Final Inference Evaluation
    if os.path.exists("best_syllogism_model.pt"):
        model.load_state_dict(torch.load("best_syllogism_model.pt"))

    # Test Set Validation Evaluation Phase
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            out = model(batch)
            all_preds.extend(out.argmax(dim=-1).cpu().numpy())
            all_targets.extend(batch.y.cpu().numpy())

    # Compile performance logs
    accuracy = accuracy_score(all_targets, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(all_targets, all_preds, average='macro')

    print("\n================ SYSTEM SUMMARY STATS ================")
    print(f"Overall Accuracy : {accuracy:.4f}")
    print(f"Macro Precision  : {precision:.4f}")
    print(f"Macro Recall     : {recall:.4f}")
    print(f"Macro F1 Score   : {f1:.4f}")
    print("======================================================")
    print(classification_report(all_targets, all_preds, target_names=['0: F', '1: T', '2: U'], zero_division=0))

if __name__ == "__main__":
    run_pipeline(train_graph_list, train_label_list, test_graph_list, test_label_list, val_graph_list, val_label_list)


Executing pipeline on device: cuda

--- Beginning Training Loop with Logging & Early Stopping ---


RuntimeError: CUDA error: CUBLAS_STATUS_ALLOC_FAILED when calling `cublasCreate(handle)`